<a href="https://colab.research.google.com/github/aryanshenoy02/Banking-Fraud-Detection/blob/main/notebooks/isolation-forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

try:
    # Skipping the first 3 rows of metadata to start directly from the table headers
    df = pd.read_csv('AryanTransactions.csv', skiprows=3)
    print("Successfully loaded by skipping the metadata rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading CSV: {e}")

In [ ]:
combined = df['Date'].astype(str) + ' ' + df['Time'].astype(str)

parsed = pd.to_datetime(
    combined.str.replace(r'\bSept\b', 'Sep', regex=True),
    format='%b %d, %Y %I:%M %p',
    errors='coerce'
)

invalid_rows = df[parsed.isna()]

display(invalid_rows)
df = df[parsed.notna()].copy()

df['Datetime'] = parsed[parsed.notna()]

In [ ]:
df['Date'] = df['Date'].str.replace(
    r'\bSept\b', 'Sep', regex=True
)

df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%b %d, %Y %I:%M %p'
)

In [ ]:
import numpy as np

# 1. Drop rows where all columns are NaN (if any exist at the end of the file)
df = df.dropna(how='all').copy()

# 2. Clean the 'Transaction Details' column
# We remove case-insensitive variations of "Paid to " and "Received from "
df['Transaction Details'] = df['Transaction Details'].str.replace(r'^(Paid to|Received from)\s+', '', case=False, regex=True)

# 3. Create a combined datetime column to sort chronologically and compute differences
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%b %d, %Y %I:%M %p')

# Sort chronologically so we can calculate the time difference forward in time
df = df.sort_values('Datetime').reset_index(drop=True)

# 4. Calculate the time difference between consecutive transactions (in hours)
df['Time_Diff_Hours'] = df['Datetime'].diff().dt.total_seconds() / 3600.0

# Display the cleaned and processed DataFrame
display(df[['Date', 'Time', 'Transaction Details', 'Transaction Type', 'Amount', 'Time_Diff_Hours']].head(10))

In [ ]:
# sort to datatime
df = df.sort_values('Datetime').reset_index(drop=True)

# Calculate cumulative transaction count per counterparty (excluding current transaction)
df['Counterparty_Transaction_Count'] = df.groupby('Transaction Details').cumcount()

# Display the results emphasizing the new column
display(df[['Datetime', 'Transaction Details', 'Counterparty_Transaction_Count']].head(20))

In [ ]:
import numpy as np

df['Log_Amount'] = np.log1p(df['Amount'])
df = df.drop(columns = ['Transaction ID', 'UTR', 'Date', 'Time'])
df.head()

In [ ]:
df[['Credit/debit instrument']]

In [9]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

df['hour'] = df['Datetime'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dayofweek'] = df['Datetime'].dt.dayofweek
df['is_new_counterparty'] = (df['Counterparty_Transaction_Count'] == 0).astype(int)
df['is_debit'] = (df['Transaction Type'] == 'DEBIT').astype(int)
#df = pd.get_dummies(df, columns=['Credit/debit instrument'], dtype=int)

# 2. Split by date
cut = int(len(df) * 0.75)
train = df.iloc[:cut].copy()
test = df.iloc[cut:].copy()

# 3. Choose features, fill NaN using TRAIN only
#inst_cols = [c for c in df.columns if c.startswith('Credit/debit instrument_')]
features = ['Log_Amount', 'Time_Diff_Hours', 'Counterparty_Transaction_Count',
            'hour_sin', 'hour_cos', 'dayofweek',
            'is_debit']

#for the 1st value since its NaN
median_gap = train['Time_Diff_Hours'].median()
train['Time_Diff_Hours'] = train['Time_Diff_Hours'].fillna(median_gap)
test['Time_Diff_Hours'] = test['Time_Diff_Hours'].fillna(median_gap)

# 4. Train and score
model = IsolationForest(n_estimators=200, random_state=42)
model.fit(train[features])

train_scores = model.decision_function(train[features])
test['score'] = model.decision_function(test[features])

# 5. Threshold and flag
threshold = np.percentile(train_scores, 1)
test['flagged'] = test['score'] < threshold

print('Before using counts on the final anomaly score : \n')
print(test['flagged'].mean())
print(test.sort_values('score').head(10)[['Transaction Details', 'Amount', 'Datetime', 'score']])

Before using counts on the final anomaly score : 

0.010309278350515464
     Transaction Details   Amount            Datetime     score
384       Mr GOVIND RAJU    42.00 2026-11-03 13:30:00 -0.123822
331          STAR BAZAAR   882.00 2026-09-16 19:50:00 -0.100886
291          STAR BAZAAR    45.00 2026-08-08 18:57:00 -0.094199
295             NAGARAJU   253.00 2026-08-16 08:38:00 -0.093052
376       Mr GOVIND RAJU    10.00 2026-10-27 10:51:00 -0.087031
302          STAR BAZAAR  1071.67 2026-08-20 12:06:00 -0.083169
311          STAR BAZAAR   719.00 2026-08-28 12:20:00 -0.083032
375        CHINCHU FEBIN   210.00 2026-10-25 12:40:00 -0.076680
385        CHINCHU FEBIN   210.00 2026-11-03 16:46:00 -0.065457
298  ANANTHANANDA SHENOY    50.00 2026-08-18 18:13:00 -0.064643


In [10]:
# 1. Flip so higher = more suspicious, then scale using TRAIN stats
train_raw = -model.decision_function(train[features])
test_raw = -model.decision_function(test[features])
lo, hi = train_raw.min(), train_raw.max()
train['anomaly'] = np.clip((train_raw - lo) / (hi - lo), 0, None)
test['anomaly'] = np.clip((test_raw - lo) / (hi - lo), 0, None)

# 2. Familiarity discount
def trust_divisor(count, k=5):
    return 1 + count / (count + k)

train['final'] = train['anomaly'] / trust_divisor(train['Counterparty_Transaction_Count'])
test['final'] = test['anomaly'] / trust_divisor(test['Counterparty_Transaction_Count'])

# 3. Threshold from train, flag test
threshold = np.percentile(train['final'], 99)
test['flagged'] = test['final'] > threshold

print('After using counts on the final anomaly score : \n')
print(test['flagged'].mean())
print(test.sort_values('final', ascending=False).head(10)
      [['Transaction Details', 'Amount', 'Datetime', 'final']])

After using counts on the final anomaly score : 

0.0
       Transaction Details  Amount            Datetime     final
295               NAGARAJU   253.0 2026-08-16 08:38:00  0.652942
298    ANANTHANANDA SHENOY    50.0 2026-08-18 18:13:00  0.560785
301  KRISHNA SATISH SHETTY    50.0 2026-08-19 13:48:00  0.478330
300             Mayank AMC   160.0 2026-08-19 13:48:00  0.473845
381            PRANATHI  J    50.0 2026-10-29 14:51:00  0.432804
366                 SWIGGY   670.0 2026-10-11 21:42:00  0.432387
382                 Lokesh    55.0 2026-10-29 16:03:00  0.415328
384         Mr GOVIND RAJU    42.0 2026-11-03 13:30:00  0.410597
374    PARKVIEW RESTAURANT   414.0 2026-10-25 12:15:00  0.396899
331            STAR BAZAAR   882.0 2026-09-16 19:50:00  0.371177


In [11]:
df.head(10)

,Transaction Details,Transaction Type,Credit/debit instrument,Amount,Datetime,Time_Diff_Hours,Counterparty_Transaction_Count,Log_Amount,hour,hour_sin,hour_cos,dayofweek,is_new_counterparty,is_debit
0,KA01AR2800,DEBIT,Paid by UPI Lite,25.00,2025-11-12 09:22:00,NaN,0,3.258097,9,0.707107,-7.071068e-01,2,1,1
1,JUBILANT FOODWORKS LTD,DEBIT,Paid by UPI Lite,125.80,2025-11-12 13:11:00,3.816667,0,4.842611,13,-0.258819,-9.659258e-01,2,1,1
2,Burger King,DEBIT,Paid by UPI Lite,41.96,2025-11-12 16:02:00,2.850000,0,3.760269,16,-0.866025,-5.000000e-01,2,1,1
3,Vighnesh (cheta) AMC,DEBIT,Paid by UPI Lite,200.00,2025-11-12 16:34:00,0.533333,0,5.303305,16,-0.866025,-5.000000e-01,2,1,1
4,Reliance Tours & Travels,DEBIT,Paid by UPI Lite,6.00,2025-11-13 11:12:00,18.633333,0,1.945910,11,0.258819,-9.659258e-01,3,1,1
5,Day to Day Super market,DEBIT,Paid by UPI Lite,40.00,2025-11-13 17:07:00,5.916667,0,3.713572,17,-0.965926,-2.588190e-01,3,1,1
6,ZEPTO MARKETPLACE PRIVATE LIMITED,DEBIT,Paid by XXXXXX8211,124.00,2025-11-15 13:05:00,43.966667,0,4.828314,13,-0.258819,-9.659258e-01,5,1,1
7,BEST CUTZ,DEBIT,Paid by XXXXXX8211,100.00,2025-11-17 18:07:00,53.033333,0,4.615121,18,-1.000000,-1.836970e-16,0,1,1
8,BirthGiver(Amma),DEBIT,Paid by XXXX1629,1.00,2025-11-18 07:40:00,13.550000,0,0.693147,7,0.965926,-2.588190e-01,1,1,1
9,Bhavya refreshment,DEBIT,Paid by XXXXXX8211,80.00,2025-11-18 11:25:00,3.750000,0,4.394449,11,0.258819,-9.659258e-01,1,1,1
